In [ ]:
%pip install openai pdfplumber python-docx pillow

from openai import OpenAI

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")
my_api_key[:5], my_api_key[-10:]
client = OpenAI(api_key=my_api_key)

### Extract Information from a PDF
Reads text with pdfplumber and sends to GPT-5-nano for summarization.

In [ ]:

pdf_path = "data/California_Employment_Offer_Letter.pdf"  

def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text.strip()

pdf_text = extract_text_from_pdf(pdf_path)
pdf_text

In [ ]:

prompt = f"Extract and summarize key details from this PDF:\n\n{pdf_text}"

#Open AI API call to summarize and extract key details from the DOCX text
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
          {"role": "system", "content": '''You summarize and extract data from PDF text. 
         Flag any items for compliance and risk
         If the number of hours of work per week is greater than 40, flag for overtime risk.
         If the compensation is less than $15 per hour, flag for minimum wage risk.
         If the contract duration is less than 3 months, flag for short-term contract risk.
         If the contract does not include a non-compete clause, flag for competitive risk.
         If the contract does not include a confidentiality clause, flag for data security risk.
         If the contract does not include a termination clause, flag for termination risk.
         If the contract does not include a dispute resolution clause, flag for legal risk.
         If the contract does not include a governing law clause, flag for jurisdictional risk.
         If the contract does not include an indemnification clause, flag for liability risk.'''},
        {"role": "user", "content": prompt}
    ]

)


In [ ]:


print("Extracted Info from PDF:\n")
print(response.choices[0].message.content)

In [ ]:
#Open AI API call to summarize and extract key details from the PDF text
prompt = f"Extract and summarize key details from this PDF:\n\n{pdf_text}"

response = ollama.chat(
    model="llama3",
    messages=[
        {"role": "system", "content": '''You summarize and extract data from PDF text. 
        Flag any items for compliance and risk
         If the number of hours of work per week is greater than 40, flag for overtime risk.
         If the compensation is less than $15 per hour, flag for minimum wage risk.
         If the contract duration is less than 3 months, flag for short-term contract risk.
         If the contract does not include a non-compete clause, flag for competitive risk.
         If the contract does not include a confidentiality clause, flag for data security risk.
         If the contract does not include a termination clause, flag for termination risk.
         If the contract does not include a dispute resolution clause, flag for legal risk.
         If the contract does not include a governing law clause, flag for jurisdictional risk.
         If the contract does not include an indemnification clause, flag for liability risk.'''},
        {"role": "user", "content": prompt}
    ]
)

print("Extracted Info from PDF:\n")
print(response['message']['content'])



### Extract Information from a DOCX (Word) File
Reads paragraphs and asks GPT-5-nano for key points.

In [ ]:

docx_path = "data/Project_List.docx" 

def extract_text_from_docx(path):
    document = docx.Document(path)
    return "\n".join([p.text for p in document.paragraphs])

docx_text = extract_text_from_docx(docx_path)

print (f"Extracted DOCX Text (first 500 chars):\n{docx_text[:500]}\n")
prompt = f"Extract key points from this DOCX content:\n\n{docx_text[:8000]}"

In [ ]:
#Open AI API call to summarize and extract key details from the DOCX text
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": '''You summarize and extract structured data from Word documents. Also, extract 
         a list of skills used in that project. Reurn as JSON with the following format:
            {
            "projects": [
                {
                "name": "Project Name",
                "description": "Brief description of the project",
                "skills": ["Skill1", "Skill2", "Skill3"]
                },
                ...
            ]
            }
        '''},
        {"role": "user", "content": prompt}
    ]
)

print("Extracted Info from DOCX:\n")
print(response.choices[0].message.content)


In [ ]:
#Ollama Extraction for DOCX text
import ollama

response = ollama.chat (
    model="llama3",
    messages=[
        {"role": "system", "content": "You summarize and extract structured data from Word documents."},
        {"role": "user", "content": prompt}
    ]
)

print("Extracted Info from DOCX:\n")
print(response["message"]["content"])
